# CAD conversion in memory

It is possible to convert from a CadQuery object to a DAGMC surface or volume mesh.

This is quicker and more concise than writing to file and reading it in again.

First import the packages needed for the cad creation ([CadQuery](https://cadquery.readthedocs.io)) and conversion ([cad_to_dagmc](https://github.com/fusion-energy/cad_to_dagmc))

In [1]:
from cad_to_dagmc import CadToDagmc
import cadquery as cq

Make some CAD geometry, we have lots of options here but to keep the example concise I've made some text.

The important thing to note here is that we have 10 separate solids in the CAD model

In [2]:

text = cq.Workplane().text(txt="cad to dagmc", fontsize=10, distance=1)
text

Fontconfig error: Cannot load default config file: No such file: (null)


2025-06-06 16:02:10.930 (   1.180s) [    77C2C0857740]vtkDemandDrivenPipeline:675    ERR| vtkCompositeDataPipeline (0x5c402bf542a0): Input port 0 of algorithm vtkTriangleFilter (0x5c402be558f0) has 0 connections but is not optional.


Next we make an instance of CadToDagmc with this geometry to prepare for conversion

In [3]:
my_model = CadToDagmc()
my_model.add_cadquery_object(
    cadquery_object=text,
    material_tags=["mat1"]* 10  # ten volumes each with with material tag "mat1"
)

10

The ```export_dagmc_h5m_file()``` function will make a surface mesh geometry of the CAD ready for use in neutronics simulations.

With openmc this DAGMC h5m file can be loading in using the ```openmc.DAGMCUniverse``` class.

This makes a 2D mesh / surface mesh / faceted geometry / triangles of the geometry faces.

With this type of geometry you can do simulations and score material tallies, cell tallies directly on the mesh volumes.

You can also overlay OpenMC meshes such as regular mesh or cylindrical mesh on top of the geometry.

The ```export_dagmc_h5m_file()``` function supports different arguments to control the mesh parameters.

For this example we will use simple settings for ```max_mesh_size``` and ```min_mesh_size```.

To see more options take a look at the [cad_to_dagmc package](https://github.com/fusion-energy/cad_to_dagmc)

In [4]:
my_model.export_dagmc_h5m_file(
    filename="dagmc.h5m",
    max_mesh_size=10,
    min_mesh_size=2,
)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [  0%] Meshing curve 2 (Line)
Info    : [  0%] Meshing curve 3 (Line)
Info    : [  0%] Meshing curve 5 (Line)
Info    : [  0%] Meshing curve 6 (Bezier)
Info    : [  0%] Meshing curve 4 (Line)
Info    : [  0%] Meshing curve 7 (Bezier)
Info    : [  0%] Meshing curve 10 (Bezier)
Info    : [  0%] Meshing curve 8 (Line)
Info    : [  0%] Meshing curve 9 (Bezier)
Info    : [  0%] Meshing curve 11 (Line)
Info    : [  0%] Meshing curve 14 (Line)
Info    : [  0%] Meshing curve 15 (Bezier)
Info    : [  0%] Meshing curve 12 (Bezier)
Info    : [  0%] Meshing curve 16 (Bezier)
Info    : [  0%] Meshing curve 13 (Bezier)
Info    : [  0%] Meshing curve 17 (Line)
Info    : [  0%] Meshing curve 18 (Bezier)
Info    : [  0%] Meshing curve 19 (Bezier)
Info    : [  0%] Meshing curve 20 (Line)
Info    : [  0%] Meshing curve 21 (Bezier)
Info    : [  0%] Meshing curve 22 (Bezier)
Info    : [  0%] Meshing curve 23 (Line)
Info    : [  0%] M

written DAGMC file dagmc.h5m


'dagmc.h5m'

DAGMC and OpenMC support another form of mesh often called an unstructured mesh.

An unstructured mesh can be overlaid over the geometry much like a regular mesh or cylindrical mesh and used to score tallies spatially.

The ```openmc.UnstructuredMesh``` class supports MOAB and LibMesh type unstructured meshes.

In this example we are making a MOAB / DAGMC unstructured mesh which is entirely made of tetrahedrals.

This is a volume mesh where the outer surface is also the same as the outer surface of the 2D mesh as we have used the same mesh parameters.

This means we have a conformal volume mesh to score on which aligns to the material / cell boundaries in the DAGMC h5m 2D mesh.

This conformal mesh is one of the advantages of UnstructuredMesh over regular meshes and cylindrical meshes

The ```export_unstructured_mesh_file()``` function will produce a vtk file than can be loaded into OpenMC using ```openmc.UnstructuredMesh```

In [5]:
my_model.export_unstructured_mesh_file(
    filename="dagmc.vtk",
    max_mesh_size=10,
    min_mesh_size=2,
)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [  0%] Meshing curve 2 (Line)
Info    : [  0%] Meshing curve 3 (Line)
Info    : [  0%] Meshing curve 8 (Line)
Info    : [  0%] Meshing curve 5 (Line)
Info    : [  0%] Meshing curve 4 (Line)
Info    : [  0%] Meshing curve 6 (Bezier)
Info    : [  0%] Meshing curve 7 (Bezier)
Info    : [  0%] Meshing curve 10 (Bezier)
Info    : [ 10%] Meshing curve 11 (Line)
Info    : [ 10%] Meshing curve 13 (Bezier)
Info    : [ 10%] Meshing curve 12 (Bezier)
Info    : [ 10%] Meshing curve 17 (Line)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 10%] Meshing curve 15 (Bezier)
Info    : [ 10%] Meshing curve 16 (Bezier)
Info    : [ 10%] Meshing curve 18 (Bezier)
Info    : [ 10%] Meshing curve 19 (Bezier)
Info    : [ 10%] Meshing curve 20 (Line)
Info    : [ 10%] Meshing curve 21 (Bezier)
Info    : [ 10%] Meshing curve 22 (Bezier)
Info    : [ 10%] Meshing curve 23 (Line)
Info    : [ 10%] Meshing curve 24 (Bezier)
Info    : [ 10%] 

Info    : Found volume 10
Info    : It. 0 - 0 nodes created - worst tet radius 2.34779 (nodes removed 0 0)
Info    : 3D refinement terminated (1073 nodes total):
Info    :  - 0 Delaunay cavities modified for star shapeness
Info    :  - 2 nodes could not be inserted
Info    :  - 194 tetrahedra created in 0.000247041 sec. (785294 tets/s)
Info    : 0 node relocations
Info    : Done meshing 3D (Wall 0.10842s, CPU 0.114531s)
Info    : Optimizing mesh...
Info    : Optimizing volume 1
Info    : Optimization starts (volume = 8.90495) with worst = 0.0213607 / average = 0.638578:
Info    : 0.00 < quality < 0.10 :         3 elements
Info    : 0.10 < quality < 0.20 :         2 elements
Info    : 0.20 < quality < 0.30 :         2 elements
Info    : 0.30 < quality < 0.40 :         4 elements
Info    : 0.40 < quality < 0.50 :        14 elements
Info    : 0.50 < quality < 0.60 :        47 elements
Info    : 0.60 < quality < 0.70 :        51 elements
Info    : 0.70 < quality < 0.80 :        51 elements

'dagmc.vtk'

Now you have two mesh files that can be used for neutronics simulations in OpenMC or other DAGMC compatible codes.

Learning objectives for this task
- Know how to produce DAGMC 2d surface mesh geometry
- Know how to produce DAGMC 3d volume mesh
- Understand the differences between the types of meshes produced by cad-to-dagmc
- know some of the arguments for customising the mesh